In [1]:
%pip install ultralytics

In [2]:
%pip install -U albumentations

In [3]:
%pip install codecarbon

In [4]:
#base_folder = '/cluster/home/sebassm/tdt4265' # Was used for idun
base_folder = r"C:\Users\sebbe\code\skole\TDT4265\project\lidar\lidar"

In [5]:
!ls /cluster/home/sebassm/tdt4265

In [5]:
lidar_yaml = f"""
path: {base_folder}
train: images/train
val: images/valid

names:
  0: pole
"""

with open(f'{base_folder}/data.yaml', 'w') as f:
    f.write(lidar_yaml)


In [4]:
from ultralytics.data.augment import Albumentations

import albumentations as A

safe_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.MotionBlur(p=0.2),
    A.RandomScale(scale_limit=0.2, p=0.3),
    A.Rotate(limit=5, p=0.2),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

Albumentations.custom = safe_transforms


In [ ]:
from ultralytics import YOLO
from codecarbon import EmissionsTracker

tracker = EmissionsTracker()

# Start tracking emissions
tracker.start()

model = YOLO('yolo11s.pt')  # Start with the small model for better quality
model.train(data=f'{base_folder}/data.yaml', epochs=40, imgsz=1024, batch=16,
    degrees=10,         # random rotation (degrees)
    translate=0.1,      # random translation
    scale=0.5,          # random scaling
    shear=3,            # random shear
    perspective=0.0,    # random perspective
    flipud=0.0,         # vertical flip probability
    fliplr=0.5,         # horizontal flip probability
    mosaic=0.5,         # mosaic augmentation probability
    mixup=0.2,          # mixup augmentation probability
    hsv_h=0.0,          # disable hue augmentation (important for data)
    hsv_s=0.0,          # disable saturation augmentation
    hsv_v=0.3,
)

# tracker stopping
tracker.stop()

In [6]:
metrics = model.val(data=f'{base_folder}/data.yaml', augment=False)
print("Precision:", metrics.box.p)
print("Recall:", metrics.box.r)
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

In [ ]:
tracker = EmissionsTracker()
tracker.start()
model.predict(source=f'{base_folder}/images/test', save=True)
tracker.stop()

In [7]:
# Load and prepare the YOLO model for prediction. This step is just for running without training the model
# You'll need to replace 'best.pt' with the path to your trained model weights file, which is usually saved in the 'runs/detect/train/weights' directory after training.
from ultralytics import YOLO
model = YOLO('best.pt')
print("YOLO model initialized and ready for prediction.")

In [9]:
from codecarbon import EmissionsTracker
tracker = EmissionsTracker()
# Start tracking emissions
tracker.start()
model.predict(
    source=f'{base_folder}/images/test',
    project=f'{base_folder}/results',
    name='run69',
    device=0,
    save_txt=True,
    save_conf=True
    )

# Stop tracking emissions
tracker.stop()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob
import os

# Find the latest training directory
train_dirs = sorted(glob.glob(f'{base_folder}/runs/train*'))
if train_dirs:
    latest_train_dir = train_dirs[-1]
    print(f"Looking at results in: {latest_train_dir}")
    
    # Display available plots
    plot_files = glob.glob(f"{latest_train_dir}/*.png") + glob.glob(f"{latest_train_dir}/*.jpg")
    print(f"Available plots: {[os.path.basename(f) for f in plot_files]}")
    
    # Display each plot
    for plot_file in plot_files:
        plt.figure(figsize=(12, 8))
        img = mpimg.imread(plot_file)
        plt.imshow(img)
        plt.title(os.path.basename(plot_file))
        plt.axis('off')
        plt.show()
else:
    print("No training directories found.")